# T11 It3: which methods are valid (no thermal training) and actually improve, on data their design never used?

**Valid** = nothing learned from thermal photos (team rule, 2026-11-14): hand rules whose few numbers were set on thermal data (allowed, like a
cut), or judges trained on satellite-made photos only. Every method is scored, unchanged, on the sets its design never used:
official TEST (194), NUDT 707, 808, 909 (300 each) - except the DINO-z veto, whose numbers were chosen in T6 while looking at all official
photos, so for it official TEST does not count. **Improvement** = no more wrong answers than today's rule on every set, 0 wrong overall would
be best, and a paired-bootstrap 95 % interval above 0 pooled over the test sets (and per set where it holds).

| method | origin | numbers set on | 
|---|---|---|
| base | today's rule without agreement: PC bar 8 / 11; Jetson bar 8, cap 10, M = 5 | team |
| DINO-z veto | T6: a candidate needs DINO z >= 1.3 or >= 15 inliers | NUDT 404 (T6) |
| early exit by inliers (Jetson) | answer after any standard query whose choice has >= 20 inliers | T8 It3 (NUDT design + official TRY) |
| two-tier rule | T8: >= 12 (PC) / 10 (Jetson) inliers, or >= 5 inliers with DINO z >= 2 and DINO top place of its query | NUDT 101-606 (T8) |
| two-tier + early exit (Jetson) | the above with the early exit at 20 inliers | T8 |
| satellite judge S1z / S5z | T11: trained on satellite-made photos only; zero-wrong cut 0.9995 (Jetson S5z 0.999); no override | cuts on NUDT 101-606 |

In [ ]:
import os, sys, gc
os.environ.setdefault("OMP_NUM_THREADS", "4")
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
import numpy as np, pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
import train.checks.candidate_judge as cj
import train.checks.scale_robust as sr
from train.checks.two_tier import decide_two_tier
from train.checks.replay import paired_bootstrap
from train.checks.holdout import answer_change
R = ROOT / "tests/Test_Iteration_11/results"; T = ROOT / "tests"
off = pd.read_csv(T / "Test_Iteration_7/results/signals/official.csv.gz")
split = pd.read_csv(T / "Test_Iteration_8/results/official_split.csv"); test_ids = set(split[split.half == "test"].photo)
off = off[off.photo.isin(test_ids)].assign(data="official_test")
parts = [off, pd.read_csv(T / "Test_Iteration_8/results/signals/nudt707.csv.gz"), pd.read_csv(T / "Test_Iteration_10/results/signals/nudt808.csv.gz"),
         pd.read_csv(T / "Test_Iteration_11/results/signals/nudt909.csv.gz")]
TEST = cj.add_context(cj.add_features(pd.concat(parts, ignore_index=True))).reset_index(drop=True)
TEST5 = cj.add_context(TEST, m=5)
SETS = ["official_test", "nudt707", "nudt808", "nudt909"]
TEST.groupby("data").photo.nunique()

In [ ]:
def bars_early(t, bar, E, jetson):
    """Today's bars, plus an early exit after any standard query whose choice has >= E inliers."""
    I = t.inliers.to_numpy()
    p = np.where(I >= E, 1.0, np.where(I >= bar[1], 0.97, np.where(I >= bar[0], 0.95, 0.0)))
    kw = dict(jetson=True, m=5) if jetson else {}
    return cj.decide(t, p, 0.9, wide_tau=0.96, override=None, min_choice=1 if jetson else 3, early_tau=0.99, **kw)
veto = lambda t: t.assign(inliers=np.where((t.z < 1.3) & (t.inliers < 15), 0.0, t.inliers))
# satellite-trained judges (T11), as in It2
SAT = pd.read_csv(R / "signals" / "sat_train.csv.gz"); items = pd.read_csv(R / "sat_photos" / "sat_train" / "items.csv")
from sklearn.cluster import KMeans
fold = KMeans(5, random_state=0, n_init=10).fit_predict(items[["east", "north"]].to_numpy())
SAT["data"] = ["sat%d" % fold[p] for p in SAT.photo]; SAT_SETS = sorted(SAT.data.unique()); SAT = cj.add_context(cj.add_features(SAT))
F = cj.FEATURES_CTX; FR = [("rli" if f == "li" else "rln22" if f == "ln22" else f) for f in F]; AUG = [0.5, 0.75, 1.0, 1.5, 2.0]
def sat_rules(t, m, cut1, cut5):
    sat = SAT if m is None else cj.add_context(SAT, m=m)
    _, s1 = cj.judge_probability(sat, SAT_SETS, F, "gbm")
    lv = sr.far_levels(sat, m); prior = dict(far_inl=float(lv.far_inl.mean()), far_n22=float(lv.far_n22.mean()))
    s5, _, st = sr.fit_judge(sat, SAT_SETS, FR, {"rli", "z", "hull", "cover9"}, AUG, lambda x: sr.add_relative(x, prior, 30, m)); del st; gc.collect()
    tr = sr.add_relative(t, prior, 30, m); kw = dict(jetson=True, m=5) if m else {}
    ee = (lambda c: max(0.98, c)) if m else (lambda c: None)
    return {"satellite judge S1z": cj.decide(t, sr.predict(s1, t, F), cut1, wide_tau=cut1, override=None, min_choice=3, early_tau=ee(cut1), **kw),
            "satellite judge S5z": cj.decide(tr, sr.predict(s5, tr, FR), cut5, wide_tau=cut5, override=None, min_choice=3, early_tau=ee(cut5), **kw)}
RULES = {}
RULES["pc"] = {"base": cj.baseline(TEST, 8, 11, min_choice=3), "DINO-z veto": cj.baseline(veto(TEST), 8, 11, min_choice=3),
               "two-tier rule": decide_two_tier(TEST, dict(a_low=5, z0=2.0, g0=0.0, i_hi=12), "pc"), **sat_rules(TEST, None, 0.9995, 0.9995)}
RULES["jetson"] = {"base": cj.baseline(TEST5, 8, 8, jetson=True, m=5), "DINO-z veto": cj.baseline(veto(TEST5), 8, 8, jetson=True, m=5),
                   "early exit by inliers (>= 20)": bars_early(TEST5, (8, 8), 20, True),
                   "two-tier rule": decide_two_tier(TEST5, dict(a_low=5, z0=2.0, g0=0.0, i_hi=10), "jetson"),
                   "two-tier + early exit 20": decide_two_tier(TEST5, dict(a_low=5, z0=2.0, g0=0.0, i_hi=10, early=20), "jetson"),
                   **sat_rules(TEST5, 5, 0.9995, 0.999)}
print("done")

In [ ]:
rows = []
for var, rules in RULES.items():
    b = rules["base"].set_index(["data", "photo"]).pts
    for name, r in rules.items():
        s = cj.summary(r); rr = r.set_index(["data", "photo"]).pts
        sets = [d for d in SETS if not (name == "DINO-z veto" and d == "official_test")]
        row = dict(var=var, method=name)
        for d in SETS:
            g = paired_bootstrap(b.loc[d].to_numpy(), rr.loc[d].reindex(b.loc[d].index).to_numpy())
            row[d] = "%.2f / %dw / %+.2f [%+.2f, %+.2f]" % (s.loc[d, "score"], s.loc[d, "wrong"], *g) + ("" if d in sets else " (seen)")
        bb = np.concatenate([b.loc[d].to_numpy() for d in sets]); nn = np.concatenate([rr.loc[d].reindex(b.loc[d].index).to_numpy() for d in sets])
        g = paired_bootstrap(bb, nn)
        row.update(pooled_gain="%+.2f [%+.2f, %+.2f]" % g, pooled_lo=g[1], wrong_total=int(sum(s.loc[d, "wrong"] for d in sets)),
                   base_wrong_total=int(sum(cj.summary(rules["base"]).loc[d, "wrong"] for d in sets)),
                   queries=round(float(np.mean([s.loc[d, "queries"] for d in sets])), 2))
        row["no_more_wrong_any_set"] = all(s.loc[d, "wrong"] <= cj.summary(rules["base"]).loc[d, "wrong"] for d in sets)
        row["verdict"] = ("reference" if name == "base" else
                          "IMPROVES (score)" if (row["no_more_wrong_any_set"] and g[1] > 0) else
                          "safer / faster, score not proven" if row["no_more_wrong_any_set"] else "no")
        rows.append(row)
V = pd.DataFrame(rows); V.to_csv(R / "it3_valid_methods.csv", index=False)
V[["var", "method", "pooled_gain", "wrong_total", "base_wrong_total", "queries", "no_more_wrong_any_set", "verdict"]]

In [ ]:
V[["var", "method"] + SETS]

## Robustness of the hand rules: inlier signals scaled x0.5 ... x2 (the four test sets) and the T9 altered photos

Wrong answers per rule (sum over the sets). The satellite judges were checked the same way in It2 (0-2 wrong at every scale).

In [ ]:
rows = []
for s in [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]:
    t = sr.scale_signals(TEST, s); t5 = sr.scale_signals(TEST, s, m=5)
    rr = {("pc", "base"): cj.baseline(t, 8, 11, min_choice=3), ("pc", "DINO-z veto"): cj.baseline(veto(t), 8, 11, min_choice=3),
          ("pc", "two-tier rule"): decide_two_tier(t, dict(a_low=5, z0=2.0, g0=0.0, i_hi=12), "pc"),
          ("jetson", "base"): cj.baseline(t5, 8, 8, jetson=True, m=5), ("jetson", "two-tier + early exit 20"): decide_two_tier(t5, dict(a_low=5, z0=2.0, g0=0.0, i_hi=10, early=20), "jetson")}
    for (var, name), r in rr.items():
        sm = cj.summary(r); rows.append(dict(var=var, rule=name, scale=s, wrong=int(sm.wrong.sum()), score=round(sm.score.mean(), 2)))
SC = pd.DataFrame(rows); SC.to_csv(R / "it3_hand_rules_scales.csv", index=False)
SC.pivot_table(index=["var", "rule"], columns="scale", values=["wrong", "score"])

In [ ]:
from train.checks.perturb import KINDS
T9 = T / "Test_Iteration_9/results/signals"
alt = cj.add_context(cj.add_features(pd.concat([pd.read_csv(T9 / f"{b}_{k}.csv.gz") for b in ["official", "nudt707"] for k in KINDS], ignore_index=True)))
alt5 = cj.add_context(alt, m=5)
rr = {("pc", "base"): cj.baseline(alt, 8, 11, min_choice=3), ("pc", "DINO-z veto"): cj.baseline(veto(alt), 8, 11, min_choice=3),
      ("pc", "two-tier rule"): decide_two_tier(alt, dict(a_low=5, z0=2.0, g0=0.0, i_hi=12), "pc"),
      ("jetson", "base"): cj.baseline(alt5, 8, 8, jetson=True, m=5), ("jetson", "two-tier + early exit 20"): decide_two_tier(alt5, dict(a_low=5, z0=2.0, g0=0.0, i_hi=10, early=20), "jetson")}
ALT = pd.DataFrame([dict(var=v, rule=n, wrong_14_runs=int(cj.summary(r).wrong.sum()), mean_score=round(cj.summary(r).score.mean(), 2)) for (v, n), r in rr.items()])
ALT.to_csv(R / "it3_hand_rules_altered.csv", index=False); ALT